# Rule-erosion sweep

One pass: 4 models x k=0..5 x 10 seeds. Everything is resumable — if the runtime dies, run
the cells again and only missing episodes are run. Results are pushed to GitHub every 10
minutes while the sweep runs and after every model, and `reports/viewer.html` renders them
straight from the pushed `episodes.jsonl` in the browser — edit the HTML any time without
rerunning anything.

In [ ]:
# 1. setup (once per runtime)
import os, sys
REPO = '/content/agent-history-effects'
if not os.path.exists(REPO):
    !git clone https://github.com/pnngocdoan/agent-history-effects {REPO}
%cd {REPO}
!git pull
sys.path.insert(0, REPO)

!pip -q install vllm

In [ ]:
# 2. GitHub push access (once per runtime; the token is never stored or printed)
from getpass import getpass
import subprocess

token = getpass('GitHub token (repo scope): ')
subprocess.run(['git', 'config', 'user.name', 'Ngoc Doan'], cwd=REPO)
subprocess.run(['git', 'config', 'user.email', 'pnngocdoan@users.noreply.github.com'], cwd=REPO)
subprocess.run(['git', 'remote', 'set-url', 'origin',
                f'https://{token}@github.com/pnngocdoan/agent-history-effects.git'], cwd=REPO)
del token

def push(msg):
    r = subprocess.run(f'git add results && git commit -m "{msg}" && git pull --rebase && git push',
                       shell=True, cwd=REPO, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip().splitlines()
    print(out[-1] if out else 'nothing to push')

In [ ]:
# 3. background saver: pushes results every 10 minutes while the sweep runs, so a dead
#    runtime loses at most 10 minutes of episodes
import threading, time

def _saver():
    while not _stop.is_set():
        _stop.wait(600)
        push('autosave: sweep in progress')

_stop = threading.Event()
threading.Thread(target=_saver, daemon=True).start()
print('autosave armed (every 10 min)')

In [ ]:
# 4. THE SWEEP — run once; rerunning only fills in what's missing
import time
from pathlib import Path
import serve

MODELS = [('Qwen/Qwen3-0.6B',   'qwen3-0.6b'),
          ('Qwen/Qwen3-1.7B',   'qwen3-1.7b'),
          ('Qwen/Qwen3-4B',     'qwen3-4b'),
          ('Qwen/Qwen3-8B-AWQ', 'qwen3-8b')]
ROUNDS, EPISODES = '0 1 2 3 4 5', 10
TARGET = 6 * EPISODES

def done(out):
    f = Path(out) / 'episodes.jsonl'
    return f.exists() and sum(1 for l in f.read_text().splitlines() if l.strip()) >= TARGET

for hf, name in MODELS:
    out = f'results/multiturn/{name}'
    if done(out):
        print(f'{name}: complete, skipping'); continue
    print(f'=== {name} ===', flush=True)
    !pkill -f api_server
    time.sleep(8)
    proc, log = serve.start_server(hf, name, log_dir=out)
    serve.wait_for_server(proc, log=log)
    !python multiturn/run_pilot.py --model {name} --rounds {ROUNDS} --episodes {EPISODES} \
        --workers 8 --out {out}
    proc.terminate()
    push(f'sweep results: {name}')      # saved to GitHub before the next model starts
_stop.set()
push('sweep complete')
print('sweep done — results on GitHub; open reports/viewer.html to inspect')

In [ ]:
# 5. tables — per model and across models
from pathlib import Path
from multiturn.review import review_summary, summarize_results

for _, name in MODELS:
    out = f'results/multiturn/{name}'
    if Path(out, 'episodes.jsonl').exists():
        print(f'\n===== {name} =====')
        review_summary(out, model=name)
summarize_results('results/multiturn')

## Inspecting episodes

`reports/viewer.html` fetches the pushed `results/.../episodes.jsonl` in the browser: nothing
to regenerate, ever. Serve it over http:

- **GitHub Pages**: repo Settings → Pages → deploy from `main`, then open
  `https://<user>.github.io/agent-history-effects/reports/viewer.html`
- **locally**: `python -m http.server` in the repo root → `http://localhost:8000/reports/viewer.html`

Edit the HTML (colors, layout, filters), push, refresh — the experiments never rerun.

In [ ]:
# 6. (optional) hand-run single episodes outside the sweep
# from multiturn.review import run_and_review
# run_and_review(k=3, model='qwen3-1.7b', out='results/multiturn/qwen3-1.7b')